In [ ]:
import pandas as pd
import re
import numpy as np

input_file = "harvard_articles.csv"
output_file = "harvard_articles_fixed.csv"

def clean_library_text(text):
    """
    Funkcja czyszcząca specyficzna dla danych bibliotecznych:
    1. Spłaszcza tekst (wiersze).
    2. Usuwa znane tagi HTML (br, i, em, p).
    3. Usuwa nawiasy < i >, ale zostawia ich zawartość (ważne dla dat i uwag katalogowych!).
    """
    if not isinstance(text, str):
        return ""
    
    # 1. Spłaszczanie
    text = re.sub(r'\s+', ' ', text).strip()
    
    # 2. Usuwanie konkretnych tagów HTML (tylko te najczęstsze, żeby nie zepsuć danych w nawiasach)
    ### usuwa całe tagi np. <br>, </i>, <p>
    text = re.sub(r'</?(br|p|i|em|b|strong)\s*/?>', ' ', text, flags=re.IGNORECASE)
    
    # 3. Usuwanie znaków < oraz > (pozostawiając treść w środku, np. <2007> -> 2007)
    text = re.sub(r'[<>]', '', text)
    
    # 4. Usuwanie znaków listy Pythona jeśli zostały w tekście ([' '])
    text = re.sub(r"^\['|'\]$", "", text) ### usuwa [' z początku i '] z końca
    
    return text.strip()

def extract_year(val):
    """Wyciąga pierwszy sensowny rok (4 cyfry) z bałaganu w pubDate"""
    if pd.isna(val):
        return np.nan
    val = str(val)
    ### szuka 4 cyfr z przedziału 1000-2029
    match = re.search(r'\b(1\d{3}|20[0-2]\d)\b', val)
    if match:
        return int(match.group(1))
    return np.nan

print(f"Wczytywanie {input_file}...")
try:
    ### low_memory=False pomaga przy większych plikach (71 MB)
    df = pd.read_csv(input_file, on_bad_lines='skip', low_memory=False)
    print(f"Wczytano {len(df):,} wierszy.")
except Exception as e:
    print(f"Błąd: {e}")
    df = pd.DataFrame()

if not df.empty:
    # 1. Usuwanie śmieci (brak abstraktu)
    print("Usuwanie wierszy bez abstraktu...")
    initial_count = len(df)
    df = df.dropna(subset=['abstract'])
    print(f"Usunięto {initial_count - len(df):,} rekordów bez treści.")

    # 2. Czyszczenie tekstów (Title, Abstract, Authors, Keyword)
    print("Czyszczenie tekstów (usuwanie tagów / klamer <>, spłaszczanie)...")
    cols_to_clean = ['title', 'abstract', 'keyword', 'authors', 'publisher']
    
    for col in cols_to_clean:
        if col in df.columns:
            df[col] = df[col].astype(str).apply(clean_library_text)
            df[col] = df[col].replace(r'^\s*$', np.nan, regex=True)
            df[col] = df[col].replace('nan', np.nan)

    # 3. Naprawa daty (pubDate)
    if 'pubDate' in df.columns:
        print("Ekstrakcja roku z pubDate...")
        df['year'] = df['pubDate'].apply(extract_year).astype('Int64')

    # 4. Deduplikacja
    ### po kolumnach title i abstract bo authors często brakuje
    dedup_cols = ['title', 'abstract']
    print(f"Deduplikacja wg: {dedup_cols}...")
    before_dedup = len(df)
    df = df.drop_duplicates(subset=dedup_cols, keep='first')
    print(f"Usunięto {before_dedup - len(df):,} duplikatów.")

    # 5. Zapis
    print(f"Zapisywanie {len(df):,} wierszy do {output_file}...")
    ### tylko sensowne kolumny do zapisu (pomijane śmieciowe id czy stary pubDate)
    final_cols = ['keyword', 'title', 'authors', 'abstract', 'year', 'publisher', 'subject']
    
    cols_to_save = [c for c in final_cols if c in df.columns]
    
    df[cols_to_save].to_csv(output_file, index=False, encoding='utf-8')
    
    # podgląd
    print("\nPróbka danych")
    print(df[['title', 'year']].head(3))
    if 'abstract' in df.columns:
        print("\nPrzykładowy abstrakt:")
        print(df['abstract'].iloc[0][:200] + "...")
